# RAG over SQuAD v2 — Chunking, Reranking, Claude & Evaluation

A more advanced version of the minimal RAG demo. This notebook builds a
retrieval-augmented generation pipeline on top of the **SQuAD v2** dataset
from HuggingFace, with several improvements over the basic example:

1. **Document chunking** — long Wikipedia passages are split into overlapping
   chunks so the retriever can find the right span instead of whole articles.
2. **Bi-encoder retrieval** — `all-MiniLM-L6-v2` embeddings + FAISS
   (`IndexFlatIP` with L2-normalized vectors → cosine similarity).
3. **Cross-encoder reranking** — top-`k` candidates are rescored with
   `cross-encoder/ms-marco-MiniLM-L-6-v2` for better precision.
4. **Generation with Claude** — the final prompt is sent to the Anthropic
   API for an actual grounded answer (with a source-citation instruction).
5. **Broad evaluation** — Recall@k, MRR, Exact-Match, token-F1 and
   semantic similarity, computed against SQuAD's gold answers.

> The notebook is **not** auto-executed. Run cells top-to-bottom. The
> Anthropic step needs an `ANTHROPIC_API_KEY` env variable.


## 1. Setup

In [ ]:
!pip install -q datasets sentence-transformers faiss-cpu tqdm anthropic scikit-learn

In [ ]:
import os
import re
import string
import collections
from typing import List, Tuple

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)


## 2. Load SQuAD v2

SQuAD v2 contains Wikipedia paragraphs (`context`) plus `(question, answers)`
pairs. Some questions are intentionally **unanswerable** — we'll filter those
out for evaluation since they require a different evaluation protocol.


In [ ]:
from datasets import load_dataset

squad = load_dataset("rajpurkar/squad_v2")
print(squad)

# Train split → corpus of unique contexts (the "documents" we retrieve from)
train_contexts = list({row["context"] for row in squad["train"]})
print(f"Unique contexts in train: {len(train_contexts):,}")

# Validation split → questions for retrieval & answer evaluation
val = squad["validation"]
print(f"Validation examples: {len(val):,}")


In [ ]:
# Peek at a single example
example = val[0]
print("Question :", example["question"])
print("Answers  :", example["answers"]["text"])
print("Context  :", example["context"][:300], "...")


## 3. Chunk Long Contexts

SQuAD passages are usually a few hundred tokens — short enough for an embedder,
but we still want a single retrieval index that handles **arbitrarily long
documents** in a real-world setup. We split each context into overlapping
windows of ~`chunk_size` words with `overlap` words of overlap so that an
answer span never falls on a chunk boundary.

We keep a back-pointer from each chunk to its source context so we can
evaluate retrieval at the **document** level later.


In [ ]:
def chunk_text(text: str, chunk_size: int = 120, overlap: int = 30) -> List[str]:
    """Split text into overlapping word-windows."""
    words = text.split()
    if len(words) <= chunk_size:
        return [text]
    step = chunk_size - overlap
    chunks = []
    for start in range(0, len(words), step):
        window = words[start : start + chunk_size]
        if not window:
            break
        chunks.append(" ".join(window))
        if start + chunk_size >= len(words):
            break
    return chunks


# Build (chunk_text, source_context_id) pairs
chunks: List[str] = []
chunk_to_doc: List[int] = []   # index into train_contexts
for doc_id, ctx in enumerate(tqdm(train_contexts, desc="chunking")):
    for piece in chunk_text(ctx):
        chunks.append(piece)
        chunk_to_doc.append(doc_id)

print(f"Chunks total: {len(chunks):,}  (avg {len(chunks)/len(train_contexts):.1f} per context)")


## 4. Encode with a Bi-Encoder

We use `all-MiniLM-L6-v2` — small (22M params), fast, and a strong baseline
for sentence-level semantic search. We L2-normalize the vectors so that an
inner-product index gives us **cosine similarity** directly.


In [ ]:
from sentence_transformers import SentenceTransformer

embed_model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

chunk_embeddings = embed_model.encode(
    chunks,
    batch_size=64,
    show_progress_bar=True,
    normalize_embeddings=True,        # cosine = inner-product on unit vectors
    convert_to_numpy=True,
).astype("float32")

print("Embeddings shape:", chunk_embeddings.shape)


## 5. Build FAISS Index

`IndexFlatIP` does an exact inner-product search. With normalized vectors that
is cosine similarity. For a few hundred thousand chunks an exact index is
plenty; for millions we'd swap in `IndexIVFFlat` or `IndexHNSWFlat`.


In [ ]:
import faiss

dim = chunk_embeddings.shape[1]
index = faiss.IndexFlatIP(dim)
index.add(chunk_embeddings)

print(f"FAISS index built — ntotal = {index.ntotal:,}")


## 6. First-Stage Retrieval

`retrieve(query, k)` returns the top-`k` chunks together with their similarity
score and the index of their source document. We over-fetch a wider candidate
pool (`fetch_k`) here so the cross-encoder reranker downstream has room to
re-order results.


In [ ]:
def retrieve(query: str, k: int = 5, fetch_k: int = 30):
    """Return list of dicts: {chunk, doc_id, score}."""
    q_emb = embed_model.encode(
        [query], normalize_embeddings=True, convert_to_numpy=True
    ).astype("float32")
    scores, idxs = index.search(q_emb, fetch_k)
    results = [
        {"chunk": chunks[i], "doc_id": chunk_to_doc[i], "score": float(s)}
        for s, i in zip(scores[0], idxs[0])
    ]
    return results[:k]


## 7. Cross-Encoder Reranker

A bi-encoder embeds the query and the chunks **separately**; a cross-encoder
sees them **together** and gives a much sharper relevance score at the cost
of speed. So we use the bi-encoder for `fetch_k=30` recall, then rerank that
shortlist with a cross-encoder and keep the top `k`.


In [ ]:
from sentence_transformers import CrossEncoder

reranker = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")


def retrieve_and_rerank(query: str, k: int = 5, fetch_k: int = 30):
    candidates = retrieve(query, k=fetch_k, fetch_k=fetch_k)
    pairs = [(query, c["chunk"]) for c in candidates]
    rerank_scores = reranker.predict(pairs)
    for c, s in zip(candidates, rerank_scores):
        c["rerank_score"] = float(s)
    candidates.sort(key=lambda c: c["rerank_score"], reverse=True)
    return candidates[:k]


## 8. Prompt Builder

We pass the reranked chunks to the LLM with explicit instructions to ground
its answer in the context, cite the chunk index it used, and say
*"I don't know"* when the context is insufficient. SQuAD v2 questions can be
unanswerable, so this last instruction matters.


In [ ]:
def build_prompt(query: str, contexts: List[dict]) -> str:
    context_block = "\n\n".join(
        f"[{i+1}] {c['chunk']}" for i, c in enumerate(contexts)
    )
    return f"""You are a careful question-answering assistant.
Answer the question using ONLY the numbered context passages below.
- Quote the passage number(s) you used in square brackets, e.g. [2].
- If the answer is not in the context, reply exactly: I don't know.
- Keep the answer short — a phrase or a single sentence.

Context:
{context_block}

Question: {query}
Answer:"""


## 9. Generate with Claude

The retrieval pipeline is now wired up to Anthropic's API. Set your key in the
environment first (don't paste it into the notebook):

```bash
export ANTHROPIC_API_KEY="sk-ant-..."
```

We use a small, cheap model (`claude-haiku-4-5`) since the work is
extractive — the heavy lifting was already done by retrieval + reranking.


In [ ]:
from anthropic import Anthropic

client = Anthropic()  # reads ANTHROPIC_API_KEY from env

GEN_MODEL = "claude-haiku-4-5"


def generate_answer(query: str, contexts: List[dict], max_tokens: int = 256) -> str:
    prompt = build_prompt(query, contexts)
    msg = client.messages.create(
        model=GEN_MODEL,
        max_tokens=max_tokens,
        messages=[{"role": "user", "content": prompt}],
    )
    return msg.content[0].text.strip()


def rag_answer(query: str, k: int = 5):
    contexts = retrieve_and_rerank(query, k=k)
    answer = generate_answer(query, contexts)
    return answer, contexts


## 10. Demo Query

In [ ]:
query = "Who was the chief architect of the Notre Dame de Paris cathedral restoration?"

answer, contexts = rag_answer(query, k=5)

print("Q:", query)
print("A:", answer)
print("\n--- Top reranked contexts ---")
for i, c in enumerate(contexts):
    print(f"\n[{i+1}]  rerank={c['rerank_score']:+.3f}  bi={c['score']:+.3f}")
    print(c["chunk"][:300], "...")


## 11. Evaluation

We evaluate two things:

**Retrieval quality** — does the retriever surface a chunk from the *correct*
source document? We measure:
- **Recall@k** — fraction of queries where the gold doc is in the top-k.
- **MRR@k** — Mean Reciprocal Rank of the gold doc.

**Answer quality** — is the LLM's final answer correct? We compute SQuAD's
official metrics on a small sample (LLM calls cost money):
- **Exact Match (EM)**
- **Token-level F1**
- **Semantic similarity** — cosine between the predicted answer and the gold
  answer in embedding space (forgiving for paraphrases).


### 11.1  Retrieval metrics on the validation set

In [ ]:
# Map every gold context to its doc_id in `train_contexts`.
# (Some validation contexts may not be in the train set — we skip those.)
context_to_doc = {ctx: i for i, ctx in enumerate(train_contexts)}

# Build evaluation tuples (question, gold_doc_id) for ANSWERABLE questions only
eval_items = []
for row in val:
    if not row["answers"]["text"]:
        continue   # unanswerable
    doc_id = context_to_doc.get(row["context"])
    if doc_id is None:
        continue
    eval_items.append((row["question"], doc_id, row["answers"]["text"]))

print(f"Evaluable questions: {len(eval_items):,}")

# Sample to keep eval cheap
SAMPLE_SIZE = 200
rng = np.random.default_rng(RANDOM_SEED)
sample_idx = rng.choice(len(eval_items), size=min(SAMPLE_SIZE, len(eval_items)), replace=False)
sample = [eval_items[i] for i in sample_idx]


In [ ]:
def eval_retrieval(sample, k_values=(1, 3, 5, 10), fetch_k=30, use_reranker=True):
    metrics = {f"recall@{k}": 0 for k in k_values}
    metrics.update({f"mrr@{k}": 0.0 for k in k_values})

    for question, gold_doc_id, _ in tqdm(sample, desc="retrieval eval"):
        if use_reranker:
            results = retrieve_and_rerank(question, k=max(k_values), fetch_k=fetch_k)
        else:
            results = retrieve(question, k=max(k_values), fetch_k=fetch_k)

        rank = next(
            (i + 1 for i, r in enumerate(results) if r["doc_id"] == gold_doc_id),
            None,
        )
        for k in k_values:
            if rank is not None and rank <= k:
                metrics[f"recall@{k}"] += 1
                metrics[f"mrr@{k}"] += 1.0 / rank

    n = len(sample)
    return {m: v / n for m, v in metrics.items()}


print("Bi-encoder only:")
bi_only = eval_retrieval(sample, use_reranker=False)
for m, v in bi_only.items():
    print(f"  {m:>10s} = {v:.3f}")

print("\nWith cross-encoder reranker:")
with_rerank = eval_retrieval(sample, use_reranker=True)
for m, v in with_rerank.items():
    print(f"  {m:>10s} = {v:.3f}")


### 11.2  Answer metrics (EM / F1 / semantic)

In [ ]:
# SQuAD-style normalization, copied from the official evaluation script.
def normalize_text(s: str) -> str:
    s = s.lower()
    s = "".join(ch for ch in s if ch not in set(string.punctuation))
    s = re.sub(r"\b(a|an|the)\b", " ", s)
    s = " ".join(s.split())
    return s


def exact_match(pred: str, golds: List[str]) -> int:
    return int(any(normalize_text(pred) == normalize_text(g) for g in golds))


def token_f1(pred: str, golds: List[str]) -> float:
    def f1(p, g):
        p_toks, g_toks = normalize_text(p).split(), normalize_text(g).split()
        if not p_toks or not g_toks:
            return float(p_toks == g_toks)
        common = collections.Counter(p_toks) & collections.Counter(g_toks)
        num_same = sum(common.values())
        if num_same == 0:
            return 0.0
        precision = num_same / len(p_toks)
        recall = num_same / len(g_toks)
        return 2 * precision * recall / (precision + recall)
    return max(f1(pred, g) for g in golds)


In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

def semantic_match(pred: str, golds: List[str]) -> float:
    pe = embed_model.encode([pred], normalize_embeddings=True)
    ge = embed_model.encode(golds, normalize_embeddings=True)
    return float(cosine_similarity(pe, ge).max())


In [ ]:
# Run end-to-end RAG on a small subset (LLM calls = $$).
ANSWER_SAMPLE_N = 25
answer_sample = sample[:ANSWER_SAMPLE_N]

rows = []
for question, gold_doc_id, golds in tqdm(answer_sample, desc="answer eval"):
    pred, _ = rag_answer(question, k=5)
    rows.append({
        "question": question,
        "gold": golds[0],
        "pred": pred,
        "EM": exact_match(pred, golds),
        "F1": token_f1(pred, golds),
        "semantic": semantic_match(pred, golds),
    })

results_df = pd.DataFrame(rows)
print(results_df[["EM", "F1", "semantic"]].mean().round(3))
results_df.head(10)


## 12. Where to Go From Here

Possible next steps for this notebook:

- **Better embeddings** — try `BAAI/bge-base-en-v1.5` or `intfloat/e5-base-v2`.
- **Hybrid retrieval** — combine BM25 (`rank_bm25`) with the dense scores.
- **Approximate index** — `IndexHNSWFlat` once the corpus passes ~1M chunks.
- **Query rewriting** — use Claude to rewrite the question before retrieval
  (HyDE / multi-query).
- **Answer verifier** — a second Claude call that checks the answer against
  the retrieved chunks and refuses if unsupported.
- **Track failures** — log the retrieval rank of the gold doc whenever the
  answer is wrong; that tells you whether to fix retrieval or generation.
